In [ ]:
# Example 1: Construct a Config and use get/set with defaults.
from donfig import Config

config = Config('myapp')
config.set({'array': {'chunk-size': '128 MiB'}, 'scheduler': 'threads'})

assert config.get('array.chunk-size') == '128 MiB'
assert config.get('scheduler') == 'threads'
assert config.get('nonexistent', default='fallback') == 'fallback'
print('get/set works:', config.get('array.chunk-size'), '|', config.get('scheduler'))


In [ ]:
# Example 2: `set` as a context manager temporarily mutates config.
from donfig import Config

config = Config('myapp')
config.set({'workers': 4})

assert config.get('workers') == 4
with config.set({'workers': 16}):
    inside = config.get('workers')
    assert inside == 16, inside
    print('inside context:', inside)

# After exiting the context, the old value is restored.
assert config.get('workers') == 4
print('after context:', config.get('workers'))


In [ ]:
# Example 3: Load configuration from a YAML file via collect_yaml.
import os
import tempfile
from donfig import Config

yaml_text = """
logging:
  level: INFO
cache:
  size: 1024
  policy: lru
"""

with tempfile.TemporaryDirectory() as tmpdir:
    yaml_path = os.path.join(tmpdir, 'myapp.yaml')
    with open(yaml_path, 'w') as f:
        f.write(yaml_text)

    config = Config('myapp', paths=[tmpdir])
    config.refresh()
    assert config.get('logging.level') == 'INFO'
    assert config.get('cache.size') == 1024
    print('loaded yaml:', config.get('cache'))


In [ ]:
# Example 4: Load configuration from environment variables.
import os
from donfig import Config

os.environ['MYAPP_FOO__BAR'] = '42'
os.environ['MYAPP_LOGGING__LEVEL'] = 'DEBUG'

try:
    config = Config('myapp')
    config.refresh()
    # donfig parses env vars as YAML scalars: '42' becomes int 42.
    foo_bar = config.get('foo.bar')
    level = config.get('logging.level')
    assert foo_bar == 42, foo_bar
    assert level == 'DEBUG', level
    print('env-loaded:', foo_bar, level)
finally:
    del os.environ['MYAPP_FOO__BAR']
    del os.environ['MYAPP_LOGGING__LEVEL']


In [ ]:
# Example 5: Defaults, update, merge, and rename round-trip.
from donfig import Config
from donfig.config_obj import merge, update

# `merge` combines dicts non-destructively; `update` mutates in place.
a = {'x': 1, 'nested': {'a': 1}}
b = {'y': 2, 'nested': {'b': 2}}
merged = merge(a, b)
assert merged == {'x': 1, 'y': 2, 'nested': {'a': 1, 'b': 2}}, merged

config = Config('myapp')
config.update_defaults({'feature': {'enabled': True, 'limit': 10}})
assert config.get('feature.enabled') is True
assert config.get('feature.limit') == 10

# Rename a deprecated key into a new location.
config.set({'old_name': 'value-x'})
config.rename({'old_name': 'new_name'})
assert config.get('new_name') == 'value-x'
assert config.get('old_name', default=None) is None
print('rename done. final keys:', sorted(config.config.keys()))
